# Constraint satisfaction problems — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def truth_rows(names):
    return [dict(zip(names, vals)) for vals in itertools.product([False, True], repeat=len(names))]
def draw_graph(nodes, edges, title='graph'):
    angles=np.linspace(0, 2*np.pi, len(nodes), endpoint=False)
    pos={n:(np.cos(a), np.sin(a)) for n,a in zip(nodes, angles)}
    plt.figure(figsize=(5,4))
    for a,b in edges:
        xa,ya=pos[a]; xb,yb=pos[b]; plt.plot([xa,xb],[ya,yb], 'k-', alpha=.6)
    for n,(x,y) in pos.items():
        plt.scatter([x],[y], s=500, zorder=3); plt.text(x,y,n,ha='center',va='center',color='white',weight='bold')
    plt.axis('equal'); plt.axis('off'); plt.title(title)
    return pos
print('setup ok')

## Search over variable assignments that satisfy constraints
A CSP has variables, domains, and constraints. These examples visualize a constraint graph, brute-force filtering, backtracking, forward checking, and min-conflicts repair.

In [ ]:
# 1) map-coloring constraint graph
nodes=['A','B','C']; edges=[('A','B'),('B','C'),('A','C')]; draw_graph(nodes,edges,'triangle map: neighbors differ')
assert len(edges)==3

In [ ]:
# 2) brute force filters assignments by all-different constraints
colors=['R','G','B']; assigns=list(itertools.product(colors, repeat=3)); ok=[len(set(a))==3 for a in assigns]
plt.figure(figsize=(6,3)); plt.bar(range(len(assigns)), ok, color=['tab:green' if x else 'tab:red' for x in ok]); plt.title('valid 3-colorings of a triangle')
assert sum(ok)==6

In [ ]:
# 3) backtracking visits fewer nodes than all assignments with early rejection
visited=0; sols=[]
def bt(partial):
    global visited
    visited+=1
    if len(partial)==3: sols.append(tuple(partial)); return
    for c in colors:
        if c not in partial: bt(partial+[c])
bt([])
plt.figure(figsize=(5,3)); plt.bar(['visited nodes','full assignments'], [visited,len(assigns)], color='tab:orange'); plt.title('early rejection prunes search')
assert visited==16 and len(sols)==6

In [ ]:
# 4) forward checking shrinks neighbor domains after A=R
dom={'A':set(colors),'B':set(colors),'C':set(colors)}; dom['A']={'R'}
for n in ['B','C']: dom[n].discard('R')
plt.figure(figsize=(5,3)); plt.bar(['A','B','C'], [len(dom[k]) for k in ['A','B','C']], color='tab:blue'); plt.title('domains after forward checking')
assert len(dom['B'])==2 and 'R' not in dom['C']

In [ ]:
# 5) min-conflicts repairs a bad assignment
assign={'A':'R','B':'R','C':'G'}
def conflicts(a): return sum(a[u]==a[v] for u,v in edges)
before=conflicts(assign); assign['B']='B'; after=conflicts(assign)
plt.figure(figsize=(5,3)); plt.bar(['before','after'], [before,after], color=['tab:red','tab:green']); plt.title('repair lowers conflicts')
assert before==1 and after==0